# Klingon Translator — Fine-Tuning NLLB-200

Fine-tunes Facebook's NLLB-200 (distilled 600M) for English ↔ Klingon translation.

**Setup:** Runtime → Change runtime type → **A100 GPU** (recommended) or T4 GPU

**Training data:** ~22,600 English-Klingon pairs from Tatoeba, OPUS, boQwI’ dictionary, paq’batlh, and curated proverbs.

**Strategy:**
- Bidirectional training (en→tlh AND tlh→en)
- SentencePiece tokenizer extension for Klingon with byte fallback and apostrophe preservation
- Transfer learning: initialize Klingon embeddings via base-tokenizer decomposition
- Pre-tokenized dataset for maximum GPU utilization
- Configurable GPU settings (batch size, precision, gradient checkpointing)
- Early stopping with configurable patience

## 1. Setup & Install

In [ ]:
!pip install -q transformers sentencepiece datasets sacrebleu accelerate pyyaml huggingface_hub

from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# Install the klingon_translator package in editable mode
!pip install -e "/content/drive/MyDrive/Klingon Translator" -q

# Log in to Hugging Face (set HF_TOKEN in Colab secrets and toggle ON)
try:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get("HF_TOKEN"))
except Exception as e:
    print(f"HF login skipped: {e}")
    print("To enable: click the key icon in the left sidebar, add HF_TOKEN, and toggle notebook access ON")

# Restart kernel so the editable install is picked up
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

## 2. Configuration

In [ ]:
from pathlib import Path

from klingon_translator.training.gpu import GPUConfig, get_gpu_info, enable_tf32_if_available, set_seed
from klingon_translator.training.trainer import TrainingConfig

# ════════════════════════════════════════════════════════════
# PROJECT PATHS
# ════════════════════════════════════════════════════════════
PROJECT_DIR = Path("/content/drive/MyDrive/Klingon Translator")
FINAL_MODEL_DIR = PROJECT_DIR / "models" / "fine-tuned"

# ════════════════════════════════════════════════════════════
# TRAINING HYPERPARAMETERS
# ════════════════════════════════════════════════════════════
training_config = TrainingConfig(
    max_epochs=25,                    # Maximum training epochs
    early_stopping_patience=3,        # Stop after N epochs without improvement
    learning_rate=5e-4,               # Peak learning rate (1e-5 to 1e-3)
    lr_scheduler="cosine",            # linear | cosine | cosine_with_restarts
                                      # polynomial | constant | constant_with_warmup
    warmup_steps=0,                   # Ramp-up steps (set 0 to use warmup_ratio)
    warmup_ratio=0.25,                # Fraction of total steps for warmup (0.0-0.5)
    weight_decay=0,                   # L2 regularization (0 to 0.1)
    seed=42,                          # Random seed for reproducibility
)

# ════════════════════════════════════════════════════════════
# GPU / HARDWARE SETTINGS
# ════════════════════════════════════════════════════════════
# Effective batch size = batch_size x gradient_accumulation_steps
gpu_config = GPUConfig(
    batch_size=128,                   # Per-device batch size
    gradient_accumulation_steps=1,    # Accumulate N steps before weight update
    use_bf16=True,                    # BFloat16 (Ampere+: A100, L4, H100)
    use_fp16=False,                   # Float16 (older GPUs: T4, V100)
    gradient_checkpointing=False,     # Saves VRAM, slower training
    dataloader_num_workers=4,         # Parallel data loading threads (0-8)
    dataloader_pin_memory=True,       # Pin memory for faster GPU transfer
)

# ════════════════════════════════════════════════════════════
# INITIALIZE
# ════════════════════════════════════════════════════════════
set_seed(training_config.seed)
tf32_enabled = enable_tf32_if_available()

gpu_info = get_gpu_info()
print(f"GPU: {gpu_info['gpu_name']} ({gpu_info['gpu_memory_gb']} GB)")
eff = gpu_config.effective_batch_size
print(f"Batch: {gpu_config.batch_size} x {gpu_config.gradient_accumulation_steps} = {eff} effective")
prec = "BF16" if gpu_config.use_bf16 else "FP16" if gpu_config.use_fp16 else "FP32"
print(f"Precision: {prec}")
if gpu_config.gradient_checkpointing:
    print("Gradient checkpointing: ON")
if tf32_enabled:
    print("TF32 enabled")

## 3. Load Data

In [ ]:
from klingon_translator.training.colab_utils import copy_data_to_local_ssd, load_jsonl

data_dir, raw_dir = copy_data_to_local_ssd(PROJECT_DIR)

train_data = load_jsonl(data_dir / "train.jsonl")
val_data = load_jsonl(data_dir / "val.jsonl")
test_data = load_jsonl(data_dir / "test.jsonl")

print(f"Train: {len(train_data):,} pairs")
print(f"Val:   {len(val_data):,} pairs")
print(f"Test:  {len(test_data):,} pairs")
print(f"Total: {len(train_data) + len(val_data) + len(test_data):,} pairs")

## 4. Extend Tokenizer with Klingon

In [ ]:
import torch
from klingon_translator.model.tokenizer import (
    collect_klingon_text,
    extend_nllb_tokenizer,
    report_tokenizer_quality,
    train_klingon_spm,
)

# Collect Klingon text from all sources
klingon_text = collect_klingon_text(data_dir=data_dir, raw_data_dir=raw_dir)
num_sentences = len(klingon_text.splitlines())
print(f"Klingon corpus: {num_sentences:,} unique sentences")

# Train SentencePiece and extend NLLB tokenizer
from pathlib import Path
spm_path = train_klingon_spm(klingon_text, output_dir=Path("/content/klingon_spm"))
tokenizer, model = extend_nllb_tokenizer(spm_path, output_dir=Path("/content/extended"))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Quality report
report = report_tokenizer_quality(tokenizer, klingon_text)
print(f"Vocab size: {len(tokenizer):,}")

## 5. Build Dataset

In [ ]:
from klingon_translator.training.dataset import BilingualDataset

train_dataset = BilingualDataset(train_data, tokenizer, seed=training_config.seed)
val_dataset = BilingualDataset(val_data, tokenizer, seed=training_config.seed)

print(f"Training examples: {len(train_dataset):,} (2x {len(train_data):,} pairs)")
print(f"Validation examples: {len(val_dataset):,} (2x {len(val_data):,} pairs)")

## 6. Train

Uses the GPU and training settings configured in cell 2.

In [ ]:
from klingon_translator.training.trainer import build_trainer

# Enable gradient checkpointing for memory-constrained GPUs
if gpu_config.gradient_checkpointing:
    model.gradient_checkpointing_enable()

trainer = build_trainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    gpu_config=gpu_config,
    training_config=training_config,
)

train_result = trainer.train()

print(f"\nTraining complete!")
print(f"  Total steps: {train_result.global_step}")
print(f"  Training loss: {train_result.training_loss:.4f}")
print(f"  Runtime: {train_result.metrics['train_runtime']:.0f}s")

## 7. Save Fine-Tuned Model

In [ ]:
from klingon_translator.training.trainer import save_model

save_model(model, tokenizer, drive_dir=FINAL_MODEL_DIR)

## 8. Evaluate

In [ ]:
from klingon_translator.training.evaluate import evaluate_test_set, run_sample_translations

# BLEU and chrF scores on held-out test set
scores = evaluate_test_set(
    test_data, model, tokenizer, batch_size=gpu_config.batch_size
)

# Sample translations with expected values
en2tlh_samples, tlh2en_samples = run_sample_translations(model, tokenizer)

## 9. Training Report

In [ ]:
from klingon_translator.training.evaluate import generate_training_report

report = generate_training_report(
    eval_scores=scores,
    sample_results_en2tlh=en2tlh_samples,
    sample_results_tlh2en=tlh2en_samples,
    train_result=train_result,
    training_config=training_config,
    gpu_config=gpu_config,
    tokenizer=tokenizer,
    train_pairs=len(train_data),
    val_pairs=len(val_data),
    test_pairs=len(test_data),
    save_path=FINAL_MODEL_DIR / "training_report.json",
)

## 10. Summary

The fine-tuned model is saved to Google Drive at:
```
Klingon Translator/models/fine-tuned/
```

To use locally:
1. Download the `models/fine-tuned/` directory from Google Drive
2. Copy to `models/nllb-klingon-extended/` in your project
3. Load with: `KlingonTranslator()`
4. Or run the Gradio app: `python app.py`

Next steps:
- Experiment with hyperparameters (cell 4) if BLEU is low
- Add more training data for better coverage
- Build Gradio web UI (Phase 6)